In [ ]:
!pip install -q langchain==0.2.11 langchain-community==0.2.10 langchain-core==0.2.23 langchain-text-splitters==0.2.2 langchain-huggingface==0.0.3 faiss-cpu pypdf bitsandbytes accelerate transformers sentence-transformers

print("✅ Installation terminée.")
print("⚠️ OBLIGATOIRE : Menu 'Exécution' > 'Redémarrer la session' maintenant.")

In [ ]:
import torch
import os
from pypdf import PdfReader

# Imports LangChain & FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Imports pour le modèle IA local
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
print("🚀 Démarrage du programme RAG Local (GPU T4)...")

# --- 1. CONFIGURATION ET LECTURE DU PDF ---

In [ ]:
chemin_pdf = 'Societe-Generale-Pilier-3_T2-2022_FR.pdf'

if not os.path.exists(chemin_pdf):
    print(f"❌ ERREUR BLOQUANTE : Le fichier '{chemin_pdf}' est introuvable.")
    print("👉 Glisse le fichier PDF dans le dossier 'Fichiers' à gauche de l'écran.")
else:
    print(f"📂 Lecture du fichier PDF...")
    lecteur = PdfReader(chemin_pdf)
    texte_brut = ''
    for page in lecteur.pages:
        t = page.extract_text()
        if t: texte_brut += t

    # DÉCOUPAGE OPTIMISÉ POUR LES TABLEAUX
    # On prend des gros morceaux (1500 caractères) pour ne pas couper les tableaux financiers
    print("✂️  Découpage du texte...")
    decoupeur = CharacterTextSplitter(
        separator="\n",
        chunk_size=1200,
        chunk_overlap=300,
        length_function=len
    )
    textes = decoupeur.split_text(texte_brut)
    print(f"   -> {len(textes)} morceaux générés.")

# --- 2. EMBEDDINGS (MÉMOIRE) ---

In [ ]:
print("🧠 Création de la base vectorielle (Indexation)...")
# On utilise le GPU (cuda) pour aller vite
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}
)
docsearch = FAISS.from_texts(textes, embeddings)
print("✅ Base de données prête.")

# --- 3. CHARGEMENT DU MODÈLE ZEPHYR 7B ---

In [ ]:
print("🤖 Chargement du modèle Zephyr 7B (Version optimisée 4-bit)...")
print("   ⏳ Patience, cela télécharge environ 5 Go (1 à 2 minutes)...")

# Configuration pour compresser le modèle et le faire tenir dans la mémoire gratuite
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "HuggingFaceH4/zephyr-7b-beta"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # La magie opère ici (compression)
    device_map="auto"
)

# Création du cerveau de réponse
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1500,      # Longueur max de la réponse
    temperature=0.1,         # 0.1 = Créativité faible (pour être factuel)
    repetition_penalty=1.15, # Pour éviter qu'il répète en boucle
    return_full_text=False   # On veut juste la réponse, pas la question répétée
)

llm_local = HuggingFacePipeline(pipeline=pipe)

# --- 4. PROMPT FRANÇAIS ---

In [ ]:
# On donne des ordres stricts au modèle pour qu'il parle français et soit pro
template_francais = """<|system|>
Tu es un expert financier. Tu analyses un rapport bancaire (Pilier 3).
Utilise UNIQUEMENT le contexte ci-dessous pour répondre.
Si la réponse n'est pas dans le texte, dis "Information non trouvée".
Réponds en français de manière synthétique.</s>
<|user|>
Contexte:
{context}

Question: {question}</s>
<|assistant|>"""

PROMPT = PromptTemplate(
    template=template_francais,
    input_variables=["context", "question"]
)

# --- 5. CRÉATION DE LA CHAÎNE ---

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
        llm=llm_local,
        chain_type="stuff",
        retriever=docsearch.as_retriever(search_kwargs={"k": 5}), # On lit 5 pages à la fois pour trouver l'info
        chain_type_kwargs={"prompt": PROMPT}
    )

# --- 6. INTERROGATION (FORMAT MANUEL) ---

In [ ]:
print("\n" + "="*50)
print("      RÉSULTATS DE L'ANALYSE")
print("="*50)

# --- REQUÊTE 1 ---

In [ ]:
requete = "quel était le montant des RWA (Expositions pondérées) à la société générale?"
print(f"\n❓ Question : {requete}")
# Note : qa_chain.invoke fait la recherche + la génération en une seule fois
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)

# --- REQUÊTE 2 ---

In [ ]:
requete = "Quels sont les principaux indicateurs de performance financière mentionnés dans le document ?"
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)

# --- REQUÊTE 3 ---

In [ ]:
requete = "Qui est l'auteur du document? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)

# --- REQUÊTE 4 ---

In [ ]:
requete = "Quels sont les risques mentionnés dans le document? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)

# --- REQUÊTE 5 ---

In [ ]:
requete = "Quel est le montant des fonds propres (Total Capitaux Propres) à la fin de la période de déclaration? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)